# College Baseball Analytics — Power Rankings & Game Predictor
**Sections:** Setup → Data → Elo Rankings → Run-Diff Model → Win Probability → Ensemble → Game Predictor → Betting → Kelly

*Google Colab + VS Code compatible. Training data: 2021–2025 NCAA Division I.*
*Data sourced from sportsdataverse (2021-2023) + ESPN scoreboard API (2024-2025) via `pull_ncaa_data.py`.*

## Section 1 — Setup & Data Loading

In [ ]:
!pip install -q xgboost scikit-learn fastai requests pyarrow python-dotenv

In [ ]:
import warnings, os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss
import xgboost as xgb
warnings.filterwarnings("ignore")

TRAIN_YEARS       = [2021, 2022, 2023, 2024, 2025]
TEST_YEAR         = 2026
ALL_YEARS         = TRAIN_YEARS + [TEST_YEAR]
STARTING_BANKROLL = 1000.0
KELLY_FRACTION    = 0.25
KELLY_CAP         = 0.10

print("Imports complete.")

In [ ]:
# ── Environment detection ─────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/CollegeBaseballAnalytics/'
else:
    import pathlib
    CACHE_DIR = str(pathlib.Path('data')) + '/'
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError:
        pass

os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Environment : {'Google Colab' if IN_COLAB else 'Local / VS Code'}")
print(f"Cache dir   : {CACHE_DIR}")

### Loading pre-built data files
Run `pull_ncaa_data.py` locally once to generate the parquet files, then upload to Drive.
The script pulls from sportsdataverse (2021-2023) and ESPN scoreboard API (2024-2025) —
no stats.ncaa.org required.

In [ ]:
GAME_PATH = CACHE_DIR + 'game_results_2021_2026.parquet'
STAT_PATH = CACHE_DIR + 'team_season_stats_2021_2026.parquet'

# ── Inline data pull (runs when parquet files are missing) ────────────────────
def _pull_sdv(years=(2021,2022,2023)):
    SDV = "https://raw.githubusercontent.com/sportsdataverse/baseballr-data/main/ncaa/schedules/csv"
    frames = []
    for yr in years:
        url = f"{SDV}/ncaa_baseball_schedule_{yr}.csv"
        print(f"  SDV {yr} ...", end="", flush=True)
        try:
            df = pd.read_csv(url)
            df = df[(df["home_team_division"]==1) & (df["away_team_division"]==1)].copy()
            df["season"]   = yr
            df["neutral"]  = df.get("neutral_site", False).fillna(False)
            df["date"]     = pd.to_datetime(df["date"], errors="coerce")
            df = df.rename(columns={"home_team_score":"home_score","away_team_score":"away_score"})
            df = df[["season","date","home_team","away_team","home_score","away_score","neutral"]]
            df["home_score"] = pd.to_numeric(df["home_score"], errors="coerce")
            df["away_score"] = pd.to_numeric(df["away_score"], errors="coerce")
            df = df.dropna(subset=["home_score","away_score"])
            frames.append(df); print(f" {len(df)} games")
        except Exception as e: print(f" FAILED: {e}")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _pull_espn(years=(2024,2025,2026)):
    from datetime import date, timedelta
    BASE = "https://site.api.espn.com/apis/site/v2/sports/baseball/college-baseball/scoreboard"
    HDR  = {"User-Agent": "Mozilla/5.0"}
    frames = []
    for yr in years:
        rows, start, end = [], date(yr,2,14), date(yr,6,28)
        d, dates = start, []
        while d <= end: dates.append(d); d += timedelta(days=1)
        print(f"  ESPN {yr}: scanning {len(dates)} dates", end="", flush=True)
        for i, d in enumerate(dates):
            try:
                r = requests.get(BASE, params={"dates":d.strftime("%Y%m%d"),"limit":200},
                                 headers=HDR, timeout=10)
                if r.status_code != 200: continue
                for ev in r.json().get("events", []):
                    comp = ev["competitions"][0]
                    if comp["status"]["type"]["name"] != "STATUS_FINAL": continue
                    ts   = comp["competitors"]
                    home = next((t for t in ts if t.get("homeAway")=="home"), ts[0])
                    away = next((t for t in ts if t.get("homeAway")=="away"), ts[1])
                    rows.append({"season":yr,"date":pd.to_datetime(ev["date"][:10]),
                                 "home_team":home["team"]["displayName"],
                                 "away_team":away["team"]["displayName"],
                                 "home_score":float(home.get("score","nan")),
                                 "away_score":float(away.get("score","nan")),
                                 "neutral":comp.get("neutralSite",False)})
            except Exception: pass
            if (i+1) % 30 == 0: print(".", end="", flush=True)
            time.sleep(0.05)
        df = pd.DataFrame(rows).dropna(subset=["home_score","away_score"])
        frames.append(df); print(f" >> {len(df)} games")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _compute_stats(games):
    home = games[["season","date","home_team","home_score","away_score","neutral"]].copy()
    home.columns = ["season","date","team","rs","ra","neutral"]; home["is_home"]=1
    away = games[["season","date","away_team","away_score","home_score","neutral"]].copy()
    away.columns = ["season","date","team","rs","ra","neutral"]; away["is_home"]=0
    long = pd.concat([home,away], ignore_index=True)
    long["win"] = (long["rs"] > long["ra"]).astype(int)
    long["rd"]  = long["rs"] - long["ra"]
    agg = (long.groupby(["team","season"])
               .agg(games=("win","count"), wins=("win","sum"),
                    rs_total=("rs","sum"), ra_total=("ra","sum"), rd_total=("rd","sum"))
               .reset_index())
    agg["losses"]           = agg["games"] - agg["wins"]
    agg["win_pct"]          = agg["wins"] / agg["games"]
    agg["avg_runs_scored"]  = agg["rs_total"] / agg["games"]
    agg["avg_runs_allowed"] = agg["ra_total"] / agg["games"]
    agg["avg_run_diff"]     = agg["rd_total"] / agg["games"]
    rs, ra = agg["rs_total"], agg["ra_total"]
    agg["pythagorean_win_pct"] = rs**1.83 / (rs**1.83 + ra**1.83).replace(0, float("nan"))
    return agg

if not os.path.exists(GAME_PATH) or not os.path.exists(STAT_PATH):
    print("Parquet files not found — pulling data now (takes ~3 min in Colab) ...")
    sdv   = _pull_sdv([2021,2022,2023])
    espn  = _pull_espn([2024,2025,2026])
    games = pd.concat([sdv, espn], ignore_index=True)
    games["neutral"] = games["neutral"].astype(bool)
    team_stats = _compute_stats(games)
    games.to_parquet(GAME_PATH, index=False)
    team_stats.to_parquet(STAT_PATH, index=False)
    print(f"Saved to {CACHE_DIR}")
else:
    games      = pd.read_parquet(GAME_PATH)
    team_stats = pd.read_parquet(STAT_PATH)

games['season']      = games['season'].astype(int)
team_stats['season'] = team_stats['season'].astype(int)

print(f"Games loaded:      {len(games):,} rows  |  seasons: {sorted(games['season'].unique())}")
print(f"Team stats loaded: {len(team_stats):,} rows  |  {team_stats['season'].nunique()} seasons")
team_stats.head(3)

## Section 1b — Feature Engineering

In [ ]:
# Pythagorean win% already computed; add league-normalised versions
def add_normalised_features(df):
    df = df.copy()
    for yr in df['season'].unique():
        mask = df['season'] == yr
        for col in ['avg_runs_scored','avg_runs_allowed','avg_run_diff','pythagorean_win_pct']:
            if col in df.columns:
                mu = df.loc[mask, col].mean()
                sd = max(float(df.loc[mask, col].std()), 1e-6)
                df.loc[mask, col + '_z'] = (df.loc[mask, col] - mu) / sd
    return df

team_stats = add_normalised_features(team_stats)

# Recent-form rolling (last 15 games win%) — computed from game log
def compute_recent_form(games, n=15):
    home = games[['season','date','home_team','home_score','away_score']].copy()
    home.columns = ['season','date','team','rs','ra']
    away = games[['season','date','away_team','away_score','home_score']].copy()
    away.columns = ['season','date','team','rs','ra']
    long = pd.concat([home, away]).sort_values('date')
    long['win'] = (long['rs'] > long['ra']).astype(int)
    long['recent_win_pct'] = (long.groupby(['team','season'])['win']
                                   .transform(lambda x: x.rolling(n, min_periods=3).mean()))
    # Season-end recent form per team
    form = (long.groupby(['team','season'])['recent_win_pct'].last().reset_index())
    return form

recent_form = compute_recent_form(games)
team_stats  = team_stats.merge(recent_form, on=['team','season'], how='left')
print("Features computed.")
print(team_stats.columns.tolist())
team_stats[['team','season','avg_runs_scored','avg_runs_allowed','pythagorean_win_pct','recent_win_pct']].head(5)

### Build Game-Level Training Matrix

In [ ]:
FEAT_COLS = ['avg_runs_scored','avg_runs_allowed','avg_run_diff',
             'pythagorean_win_pct','win_pct','recent_win_pct',
             'avg_runs_scored_z','avg_runs_allowed_z','pythagorean_win_pct_z']
FEAT_COLS = [c for c in FEAT_COLS if c in team_stats.columns]

def build_game_matrix(games, stats):
    idx = stats.set_index(['team','season'])
    rows = []
    for _, g in games.iterrows():
        yr = int(g['season'])
        ht, at = g['home_team'], g['away_team']
        def feat(t):
            key = (t, yr)
            return idx.loc[key, FEAT_COLS] if key in idx.index else pd.Series(np.nan, index=FEAT_COLS)
        hf, af = feat(ht), feat(at)
        row = {
            'season':   yr,
            'home':     ht,
            'away':     at,
            'is_home':  1,
            'win':      int(g['home_score'] > g['away_score']),
            'run_diff': float(g['home_score'] - g['away_score']),
            'neutral':  int(g.get('neutral', False)),
        }
        for c in FEAT_COLS:
            row[f'h_{c}'] = hf.get(c, np.nan)
            row[f'a_{c}'] = af.get(c, np.nan)
            row[f'd_{c}'] = hf.get(c, np.nan) - af.get(c, np.nan)
        rows.append(row)
    return pd.DataFrame(rows)

print("Building game matrix (may take ~30s) …")
game_matrix = build_game_matrix(games, team_stats)
game_matrix.dropna(subset=['win','run_diff'], inplace=True)
print(f"Game matrix: {game_matrix.shape}")
game_matrix.head(3)

## Section 2 — Power Rankings (Elo + Composite)

In [ ]:
ELO_K    = 20   # FiveThirtyEight-style: lower K; MOVM multiplier handles magnitude
ELO_HOME = 35   # home advantage in Elo points (college, ~35 pts empirically)
ELO_INIT = 1500

# Conference tier multipliers — scales Elo K-factor so beating elite programs
# moves the needle more than beating low-major opponents.
CONF_TIER = {
    # Elite
    'SEC': 1.00, 'ACC': 1.00, 'Big 12': 1.00,
    # High mid-major
    'Sun Belt': 0.90, 'Mountain West': 0.90, 'Big Ten': 0.90,
    'Pac-12': 0.90, 'American Athletic': 0.90,
    # Mid-major
    'Conference USA': 0.78, 'WAC': 0.78, 'MAC': 0.78,
    'Missouri Valley': 0.78, 'Atlantic 10': 0.78,
    # Low mid-major / small conference
    'Ohio Valley': 0.65, 'ASUN': 0.65, 'Big South': 0.65,
    'Southern': 0.65, 'Northeast': 0.65, 'Patriot': 0.65,
    'MAAC': 0.65, 'America East': 0.65, 'Southland': 0.65,
}
CONF_DEFAULT = 0.72

# ESPN display name → conference (covers ~300 D-I baseball programs)
TEAM_CONF = {
    # SEC
    'Alabama Crimson Tide':'SEC','Arkansas Razorbacks':'SEC','Auburn Tigers':'SEC',
    'Florida Gators':'SEC','Georgia Bulldogs':'SEC','Kentucky Wildcats':'SEC',
    'LSU Tigers':'SEC','Ole Miss Rebels':'SEC','Mississippi State Bulldogs':'SEC',
    'Missouri Tigers':'SEC','South Carolina Gamecocks':'SEC',
    'Tennessee Volunteers':'SEC','Texas A&M Aggies':'SEC',
    'Vanderbilt Commodores':'SEC','Oklahoma Sooners':'SEC','Texas Longhorns':'SEC',
    # ACC
    'Clemson Tigers':'ACC','Duke Blue Devils':'ACC','Florida State Seminoles':'ACC',
    'Georgia Tech Yellow Jackets':'ACC','Louisville Cardinals':'ACC',
    'Miami Hurricanes':'ACC','NC State Wolfpack':'ACC','North Carolina Tar Heels':'ACC',
    'Notre Dame Fighting Irish':'ACC','Virginia Cavaliers':'ACC',
    'Virginia Tech Hokies':'ACC','Wake Forest Demon Deacons':'ACC',
    'Boston College Eagles':'ACC','Pittsburgh Panthers':'ACC',
    'Syracuse Orange':'ACC','Stanford Cardinal':'ACC',
    'Cal Bears':'ACC','SMU Mustangs':'ACC',
    # Big 12
    'Baylor Bears':'Big 12','BYU Cougars':'Big 12','Cincinnati Bearcats':'Big 12',
    'Houston Cougars':'Big 12','Iowa State Cyclones':'Big 12','Kansas Jayhawks':'Big 12',
    'Kansas State Wildcats':'Big 12','Oklahoma State Cowboys':'Big 12',
    'TCU Horned Frogs':'Big 12','Texas Tech Red Raiders':'Big 12',
    'UCF Knights':'Big 12','West Virginia Mountaineers':'Big 12',
    'Arizona Wildcats':'Big 12','Arizona State Sun Devils':'Big 12',
    'Colorado Buffaloes':'Big 12','Utah Utes':'Big 12',
    # Sun Belt
    'Appalachian State Mountaineers':'Sun Belt','Arkansas State Red Wolves':'Sun Belt',
    'Coastal Carolina Chanticleers':'Sun Belt','Georgia Southern Eagles':'Sun Belt',
    'Georgia State Panthers':'Sun Belt','James Madison Dukes':'Sun Belt',
    "Louisiana Ragin' Cajuns":'Sun Belt','Louisiana Ragin Cajuns':'Sun Belt',
    'Louisiana Monroe Warhawks':'Sun Belt','Marshall Thundering Herd':'Sun Belt',
    'Old Dominion Monarchs':'Sun Belt','South Alabama Jaguars':'Sun Belt',
    'Southern Miss Golden Eagles':'Sun Belt','Texas State Bobcats':'Sun Belt',
    'Troy Trojans':'Sun Belt',
    # AAC
    'Charlotte 49ers':'American Athletic','East Carolina Pirates':'American Athletic',
    'Florida Atlantic Owls':'American Athletic','Memphis Tigers':'American Athletic',
    'Rice Owls':'American Athletic','South Florida Bulls':'American Athletic',
    'Temple Owls':'American Athletic','Tulane Green Wave':'American Athletic',
    'Tulsa Golden Hurricane':'American Athletic','UAB Blazers':'American Athletic',
    'Wichita State Shockers':'American Athletic','Navy Midshipmen':'American Athletic',
    # Big Ten
    'Indiana Hoosiers':'Big Ten','Illinois Fighting Illini':'Big Ten',
    'Maryland Terrapins':'Big Ten','Michigan Wolverines':'Big Ten',
    'Michigan State Spartans':'Big Ten','Minnesota Golden Gophers':'Big Ten',
    'Nebraska Cornhuskers':'Big Ten','Northwestern Wildcats':'Big Ten',
    'Ohio State Buckeyes':'Big Ten','Penn State Nittany Lions':'Big Ten',
    'Purdue Boilermakers':'Big Ten','Rutgers Scarlet Knights':'Big Ten',
    # Pac-12 remnants
    'Oregon State Beavers':'Pac-12','UCLA Bruins':'Pac-12',
    'Washington Huskies':'Pac-12','Oregon Ducks':'Pac-12',
    # Mountain West
    'Air Force Falcons':'Mountain West','Fresno State Bulldogs':'Mountain West',
    'Nevada Wolf Pack':'Mountain West','UNLV Rebels':'Mountain West',
    'Utah State Aggies':'Mountain West','San Diego State Aztecs':'Mountain West',
    'New Mexico Lobos':'Mountain West','Wyoming Cowboys':'Mountain West',
    'Boise State Broncos':'Mountain West',
    # Conference USA
    'Jacksonville State Gamecocks':'Conference USA','Liberty Flames':'Conference USA',
    'Middle Tennessee Blue Raiders':'Conference USA',
    'New Mexico State Aggies':'Conference USA','UTEP Miners':'Conference USA',
    'Western Kentucky Hilltoppers':'Conference USA','Sam Houston Bearkats':'Conference USA',
    'FIU Panthers':'Conference USA','Louisiana Tech Bulldogs':'Conference USA',
    'UTSA Roadrunners':'Conference USA','Kennesaw State Owls':'Conference USA',
    'Florida International Panthers':'Conference USA',
    # WAC
    'California Baptist Lancers':'WAC','Cal Baptist Lancers':'WAC',
    'Grand Canyon Antelopes':'WAC','Sacramento State Hornets':'WAC',
    'Tarleton State Texans':'WAC','Utah Tech Trailblazers':'WAC',
    'Utah Valley Wolverines':'WAC','Seattle Redhawks':'WAC',
    'Southern Utah Thunderbirds':'WAC','Abilene Christian Wildcats':'WAC',
    "Stephen F. Austin Lumberjacks":'WAC',
    # MAC
    'Bowling Green Falcons':'MAC','Ball State Cardinals':'MAC',
    'Central Michigan Chippewas':'MAC','Eastern Michigan Eagles':'MAC',
    'Kent State Golden Flashes':'MAC','Miami RedHawks':'MAC',
    'Northern Illinois Huskies':'MAC','Ohio Bobcats':'MAC',
    'Toledo Rockets':'MAC','Western Michigan Broncos':'MAC',
    'Akron Zips':'MAC','Buffalo Bulls':'MAC',
    # Missouri Valley
    'Dallas Baptist Patriots':'Missouri Valley','Illinois State Redbirds':'Missouri Valley',
    'Indiana State Sycamores':'Missouri Valley','Missouri State Bears':'Missouri Valley',
    'Southern Illinois Salukis':'Missouri Valley','Bradley Braves':'Missouri Valley',
    'Valparaiso Beacons':'Missouri Valley',
    # OVC
    'Morehead State Eagles':'Ohio Valley','Austin Peay Governors':'Ohio Valley',
    'Eastern Illinois Panthers':'Ohio Valley','Eastern Kentucky Colonels':'Ohio Valley',
    'Murray State Racers':'Ohio Valley','SE Missouri State Redhawks':'Ohio Valley',
    'Tennessee Tech Golden Eagles':'Ohio Valley','UT Martin Skyhawks':'Ohio Valley',
    'SIU Edwardsville Cougars':'Ohio Valley','Lindenwood Lions':'Ohio Valley',
    'Bellarmine Knights':'Ohio Valley',
    # ASUN
    'Central Arkansas Bears':'ASUN','Jacksonville Dolphins':'ASUN',
    'Lipscomb Bisons':'ASUN','North Alabama Lions':'ASUN',
    'North Florida Ospreys':'ASUN','Northern Kentucky Norse':'ASUN',
    'Stetson Hatters':'ASUN','Queens Royals':'ASUN',
    # Big South
    'Campbell Fighting Camels':'Big South','Charleston Southern Buccaneers':'Big South',
    "Gardner-Webb Runnin' Bulldogs":'Big South','High Point Panthers':'Big South',
    'Longwood Lancers':'Big South','Presbyterian Blue Hose':'Big South',
    'Radford Highlanders':'Big South','UNC Asheville Bulldogs':'Big South',
    'USC Upstate Spartans':'Big South','Winthrop Eagles':'Big South',
    # Southern
    'ETSU Buccaneers':'Southern','East Tennessee State Buccaneers':'Southern',
    'Furman Paladins':'Southern','Mercer Bears':'Southern',
    'Samford Bulldogs':'Southern','The Citadel Bulldogs':'Southern',
    'VMI Keydets':'Southern','Western Carolina Catamounts':'Southern',
    'Wofford Terriers':'Southern','Chattanooga Mocs':'Southern',
    # NEC / Patriot / MAAC / America East
    'Bryant Bulldogs':'Northeast','Sacred Heart Pioneers':'Northeast',
    'Wagner Seahawks':'Northeast',
    'Army Black Knights':'Patriot','Bucknell Bison':'Patriot',
    'Holy Cross Crusaders':'Patriot','Lafayette Leopards':'Patriot',
    'Lehigh Mountain Hawks':'Patriot',
    'Fairfield Stags':'MAAC','Manhattan Jaspers':'MAAC','Marist Red Foxes':'MAAC',
    'Niagara Purple Eagles':'MAAC','Quinnipiac Bobcats':'MAAC',
    'Rider Broncs':'MAAC','Siena Saints':'MAAC',
    'Albany Great Danes':'America East','Maine Black Bears':'America East',
    'Stony Brook Seawolves':'America East','Vermont Catamounts':'America East',
    'UMass Lowell River Hawks':'America East','Binghamton Bearcats':'America East',
}

def get_conf_tier(team):
    return CONF_TIER.get(TEAM_CONF.get(team, ''), CONF_DEFAULT)

def expected_win(elo_a, elo_b):
    return 1.0 / (1.0 + 10 ** ((elo_b - elo_a) / 400))

def margin_mult_538(run_diff, elo_diff_pre):
    """
    FiveThirtyEight baseball MOVM formula.
    Autocorrelation correction: strong favorites gain less Elo for expected blowouts.
    elo_diff_pre = winning team Elo - losing team Elo (before the game, always positive).
    """
    movm = np.log(abs(run_diff) + 1) * 2.2
    # Autocorrelation correction — dampens margin value when elo gap was already large
    correction = 2.2 / (elo_diff_pre * 0.001 + 2.2)
    return movm * correction

ELO_MEAN_REVERT  = 0.75   # carry fraction from prior season (0.25 reverts to mean)
ELO_REVERT_TO    = 1505   # long-run mean (slightly above 1500 to account for new teams)

def compute_elo(games):
    """
    FiveThirtyEight-style Elo:
    - K=20 (lower than naive; MOVM handles magnitude)
    - Autocorrelation correction in MOVM (see margin_mult_538)
    - Season mean-reversion: Elo *= 0.75 + 0.25 * 1505 at season boundary
    - Conference K-tier multiplier preserved
    """
    elo = {}
    cur_season = None
    games_s = games.sort_values(['season', 'date']).dropna(subset=['date'])
    for _, g in games_s.iterrows():
        yr = int(g['season'])
        # Mean-revert all ratings at the start of each new season
        if yr != cur_season:
            if cur_season is not None:
                elo = {t: ELO_MEAN_REVERT * v + (1 - ELO_MEAN_REVERT) * ELO_REVERT_TO
                       for t, v in elo.items()}
            cur_season = yr

        ht, at = g['home_team'], g['away_team']
        base_h = elo.get(ht, ELO_INIT)
        base_a = elo.get(at, ELO_INIT)

        # Apply home advantage before computing expected win
        eh = base_h + (ELO_HOME if not g.get('neutral', False) else 0)
        ea = base_a
        exp = expected_win(eh, ea)
        actual = 1 if g['home_score'] > g['away_score'] else 0

        # Pre-game Elo diff from winner's perspective (always positive, used for MOVM)
        rd = g['home_score'] - g['away_score']
        if rd > 0:
            winner_elo, loser_elo = eh, ea
        elif rd < 0:
            winner_elo, loser_elo = ea, eh
        else:
            winner_elo, loser_elo = eh, ea
        elo_diff_pre = max(winner_elo - loser_elo, 0)

        mult  = margin_mult_538(rd, elo_diff_pre)
        tier  = (get_conf_tier(ht) + get_conf_tier(at)) / 2
        delta = ELO_K * tier * mult * (actual - exp)

        elo[ht] = base_h + delta
        elo[at] = base_a - delta

    return pd.DataFrame({'team': list(elo.keys()), 'elo': list(elo.values())})

elo_df = compute_elo(games)
print(f"Elo (FiveThirtyEight-style) computed for {len(elo_df)} teams")
elo_df.sort_values('elo', ascending=False).head(10)


In [ ]:
def compute_sos(games, elo):
    """Average Elo of every opponent faced — higher means harder schedule."""
    elo_lk = elo.set_index('team')['elo']
    home = games[['season','home_team','away_team']].rename(
        columns={'home_team':'team','away_team':'opp'})
    away = games[['season','away_team','home_team']].rename(
        columns={'away_team':'team','home_team':'opp'})
    long = pd.concat([home, away], ignore_index=True)
    long['opp_elo'] = long['opp'].map(elo_lk).fillna(ELO_INIT)
    return (long.groupby(['team','season'])['opp_elo']
               .mean().reset_index()
               .rename(columns={'opp_elo':'avg_opp_elo'}))

sos_df = compute_sos(games, elo_df)

def build_rankings(stats, elo, sos, year=TEST_YEAR):
    df = stats[stats['season'] == year].copy()
    df = df.merge(elo, on='team', how='left')
    df = df.merge(sos[sos['season'] == year][['team','avg_opp_elo']], on='team', how='left')
    df['elo']         = df['elo'].fillna(ELO_INIT)
    df['avg_opp_elo'] = df['avg_opp_elo'].fillna(df['avg_opp_elo'].mean())

    def z(s): return (s - s.mean()) / max(s.std(), 1e-6)

    score = pd.Series(0.0, index=df.index)
    for col, wt, higher in [
        ('pythagorean_win_pct', 0.25, True),   # was 0.30
        ('avg_run_diff',        0.20, True),    # was 0.25
        ('elo',                 0.20, True),    # was 0.25
        ('avg_runs_scored',     0.10, True),
        ('avg_runs_allowed',    0.10, False),
        ('avg_opp_elo',         0.15, True),    # new SOS component
    ]:
        if col in df.columns:
            score += wt * (z(df[col]) if higher else -z(df[col]))

    df['power_score'] = score
    df['conference']  = df['team'].map(TEAM_CONF).fillna('Unknown')
    df['rank'] = df['power_score'].rank(ascending=False).astype(int)
    return df.sort_values('rank')

rankings = build_rankings(team_stats, elo_df, sos_df, year=TEST_YEAR)
show = ['rank','team','conference','elo','avg_opp_elo','pythagorean_win_pct',
        'avg_run_diff','power_score']
show = [c for c in show if c in rankings.columns]
print(f"=== {TEST_YEAR} College Baseball Power Rankings (Top 25) ===")
rankings[show].head(25).style.format({
    'elo':'{:.0f}','avg_opp_elo':'{:.0f}','pythagorean_win_pct':'{:.3f}',
    'avg_run_diff':'{:+.2f}','power_score':'{:.3f}'
})

In [ ]:
print("Exact team names for Top 25 (copy/paste into predict_game):")
for _, r in rankings.head(25).iterrows():
    conf = r.get('conference','?')
    print(f"  #{r['rank']:2d}  {r['team']}  [{conf}]")

In [ ]:
top25 = rankings.head(25)
fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top25)))[::-1]
ax.barh(top25['team'][::-1], top25['power_score'][::-1], color=colors[::-1])
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('Composite Power Score')
ax.set_title(f'{TEST_YEAR} College Baseball Power Rankings — Top 25', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Section 2b — Team Profiles & Comparison

### Usage

```python
# Single team deep-dive
team_profile("Tennessee Volunteers")

# Side-by-side comparison with radar chart
compare_teams("Tennessee Volunteers", "Morehead State Eagles")
```


In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────
PROFILE_METRICS = [
    ('pythagorean_win_pct', 'Pythag W%',   True),
    ('avg_runs_scored',     'Off (RS/G)',   True),
    ('avg_runs_allowed',    'Def (RA/G)',   False),
    ('avg_run_diff',        'Run Diff',     True),
    ('elo',                 'Elo',          True),
    ('avg_opp_elo',         'SOS',          True),
    ('recent_win_pct',      'Recent Form',  True),
    ('power_score',         'Power Score',  True),
]

def _get_row(team, year=TEST_YEAR):
    rk = rankings[rankings['team'] == team]
    if not rk.empty:
        return rk.iloc[0]
    s = team_stats[(team_stats['team']==team)&(team_stats['season']==year)]
    if s.empty: return None
    row = s.iloc[0].copy()
    row['elo']         = elo_df.set_index('team')['elo'].get(team, ELO_INIT)
    sos_val            = sos_df[(sos_df['team']==team)&(sos_df['season']==year)]
    row['avg_opp_elo'] = sos_val['avg_opp_elo'].values[0] if not sos_val.empty else ELO_INIT
    row['power_score'] = float('nan')
    row['rank']        = float('nan')
    return row

def _percentile(col, val, higher=True):
    vals = rankings[col].dropna()
    if vals.empty or pd.isna(val): return 0
    return int(((vals < val).sum() / len(vals)) * 100) if higher else int(((vals > val).sum() / len(vals)) * 100)

def _bar(p, width=20):
    filled = int(p / 100 * width)
    return '[' + '#'*filled + '.'*(width-filled) + ']'

# ── team_profile ──────────────────────────────────────────────────────────────
def team_profile(team, year=TEST_YEAR):
    row = _get_row(team, year)
    if row is None:
        print("No data for '{}' in {}.".format(team, year)); return

    conf     = TEAM_CONF.get(team, 'Unknown')
    tier     = CONF_TIER.get(conf, CONF_DEFAULT)
    n_teams  = len(rankings)
    rank_str = '#{}'.format(int(row['rank'])) if not pd.isna(row.get('rank', float('nan'))) else 'N/A'
    wins     = int(row.get('wins',   0))
    losses   = int(row.get('losses', 0))

    sep = '=' * 58
    print('\n' + sep)
    print('  ' + team)
    print('  {}  |  Conf tier: {:.2f}  |  Rank: {} / {}'.format(conf, tier, rank_str, n_teams))
    print(sep)
    print('  Record          : {}-{}  ({:.3f})'.format(wins, losses, row.get('win_pct', 0)))
    print()

    labels = {
        'pythagorean_win_pct': ('Pythagorean W%  ', '{:.3f}'.format(row.get('pythagorean_win_pct', 0))),
        'avg_runs_scored':     ('Runs Scored/G   ', '{:.2f}'.format(row.get('avg_runs_scored', 0))),
        'avg_runs_allowed':    ('Runs Allowed/G  ', '{:.2f}'.format(row.get('avg_runs_allowed', 0))),
        'avg_run_diff':        ('Run Diff/G      ', '{:+.2f}'.format(row.get('avg_run_diff', 0))),
        'elo':                 ('Elo Rating      ', '{:.0f}'.format(row.get('elo', ELO_INIT))),
        'avg_opp_elo':         ('SOS (Opp Elo)   ', '{:.0f}'.format(row.get('avg_opp_elo', ELO_INIT))),
        'recent_win_pct':      ('Recent Form L15 ', '{:.3f}'.format(row.get('recent_win_pct', 0))),
        'power_score':         ('Power Score     ', '{:.3f}'.format(row.get('power_score', 0))),
    }

    for col, higher in [(c, h) for c, _, h in PROFILE_METRICS]:
        val = row.get(col, float('nan'))
        if pd.isna(val): continue
        lbl, fmt_val = labels[col]
        p = _percentile(col, val, higher)
        print('  {}: {:>8}   P{:3d}  {}'.format(lbl, fmt_val, p, _bar(p)))
    print(sep)

# ── compare_teams ─────────────────────────────────────────────────────────────
def compare_teams(team_a, team_b, year=TEST_YEAR):
    ra = _get_row(team_a, year)
    rb = _get_row(team_b, year)
    if ra is None: print("No data for '{}'".format(team_a)); return
    if rb is None: print("No data for '{}'".format(team_b)); return

    conf_a = TEAM_CONF.get(team_a, 'Unknown')
    conf_b = TEAM_CONF.get(team_b, 'Unknown')

    h2h  = games[
        ((games['home_team']==team_a)&(games['away_team']==team_b)) |
        ((games['home_team']==team_b)&(games['away_team']==team_a))
    ].copy()
    a_wins = (((h2h['home_team']==team_a)&(h2h['home_score']>h2h['away_score'])) |
               ((h2h['away_team']==team_a)&(h2h['away_score']>h2h['home_score']))).sum()
    b_wins = len(h2h) - a_wins

    col_w = 22
    sep   = '=' * 66
    print('\n' + sep)
    print('  {:<22}       {:<{w}}  {:<{w}}'.format('METRIC', team_a[:col_w], team_b[:col_w], w=col_w))
    print('  {:<22}       [{:<18}]  [{:<18}]'.format('', conf_a[:18], conf_b[:18]))
    print(sep)

    display_metrics = [
        ('pythagorean_win_pct', 'Pythagorean W%',  True,  '{:.3f}'),
        ('avg_runs_scored',     'Runs Scored/G',    True,  '{:.2f}'),
        ('avg_runs_allowed',    'Runs Allowed/G',   False, '{:.2f}'),
        ('avg_run_diff',        'Run Diff/G',       True,  '{:+.2f}'),
        ('elo',                 'Elo Rating',        True,  '{:.0f}'),
        ('avg_opp_elo',         'SOS (Opp Elo)',     True,  '{:.0f}'),
        ('recent_win_pct',      'Recent Form L15',   True,  '{:.3f}'),
        ('power_score',         'Power Score',       True,  '{:.3f}'),
    ]

    for col, label, higher, fmt_str in display_metrics:
        va = ra.get(col, float('nan'))
        vb = rb.get(col, float('nan'))
        if pd.isna(va) and pd.isna(vb): continue
        a_better = not pd.isna(va) and not pd.isna(vb) and ((va > vb) if higher else (va < vb))
        b_better = not pd.isna(va) and not pd.isna(vb) and ((vb > va) if higher else (vb < va))
        fa = fmt_str.format(va) + (' <<' if a_better else '   ')
        fb = fmt_str.format(vb) + (' <<' if b_better else '   ')
        print('  {:<22}       {:<{w}}  {:<{w}}'.format(label, fa, fb, w=col_w))

    print(sep)
    print('  H2H Record (all seasons):  {} {}-{} {}'.format(team_a[:20], a_wins, b_wins, team_b[:20]))
    print(sep)

    radar_cols   = ['pythagorean_win_pct','avg_runs_scored','avg_runs_allowed',
                    'avg_run_diff','elo','avg_opp_elo','recent_win_pct','power_score']
    radar_labels = ['Pythag W%','Offense','Defense\n(inv)','Run Diff',
                    'Elo','SOS','Recent\nForm','Power\nScore']
    radar_higher = [True, True, False, True, True, True, True, True]

    def norm(col, val, higher):
        vals = rankings[col].dropna()
        if vals.empty or pd.isna(val): return 0.5
        mn, mx = vals.min(), vals.max()
        if mx == mn: return 0.5
        n = (val - mn) / (mx - mn)
        return n if higher else 1.0 - n

    vals_a = [norm(c, ra.get(c, float('nan')), h) for c, h in zip(radar_cols, radar_higher)]
    vals_b = [norm(c, rb.get(c, float('nan')), h) for c, h in zip(radar_cols, radar_higher)]

    N      = len(radar_cols)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]; vals_a += vals_a[:1]; vals_b += vals_b[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    ax.plot(angles, vals_a, 'o-', linewidth=2, color='royalblue', label=team_a)
    ax.fill(angles, vals_a, alpha=0.15, color='royalblue')
    ax.plot(angles, vals_b, 'o-', linewidth=2, color='darkorange', label=team_b)
    ax.fill(angles, vals_b, alpha=0.15, color='darkorange')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, size=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['25th', '50th', '75th'], size=7, color='gray')
    ax.set_title(team_a + '\nvs\n' + team_b, size=11, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=9)
    plt.tight_layout()
    plt.show()

# ── sample calls ──────────────────────────────────────────────────────────────
team_profile("Tennessee Volunteers")
compare_teams("Tennessee Volunteers", "Morehead State Eagles")

## Section 3 — Run Differential Prediction (XGBoost)

In [ ]:
DIFF_FEATS = [c for c in game_matrix.columns if c.startswith('d_')] + ['is_home','neutral']
DIFF_FEATS = [c for c in DIFF_FEATS if c in game_matrix.columns]

def split_train_test(matrix, target, feats):
    df = matrix.dropna(subset=[target]+feats).copy()
    tr = df[df['season'].isin(TRAIN_YEARS)]
    te = df[df['season'] == TEST_YEAR]
    return tr[feats].values, tr[target].values, te[feats].values, te[target].values, tr, te

X_tr, y_tr, X_te, y_te, tr_df, te_df = split_train_test(game_matrix, 'run_diff', DIFF_FEATS)
print(f"Train: {len(X_tr):,} games  |  Test: {len(X_te):,} games  |  Features: {len(DIFF_FEATS)}")

reg = xgb.XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                        reg_alpha=0.5, random_state=42, n_jobs=-1)
reg.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

from sklearn.metrics import mean_absolute_error, r2_score
preds_reg = reg.predict(X_te)
print(f"MAE: {mean_absolute_error(y_te, preds_reg):.2f} runs  |  R2: {r2_score(y_te, preds_reg):.3f}")

fi = pd.Series(reg.feature_importances_, index=DIFF_FEATS).sort_values(ascending=False)
fi.head(12).plot(kind='bar', figsize=(10,4), color='steelblue', title='Feature Importances (Run Diff)')
plt.tight_layout(); plt.show()

## Section 4a — Win Probability (XGBoost Classifier)

In [ ]:
X_tr_c, y_tr_c, X_te_c, y_te_c, _, _ = split_train_test(game_matrix, 'win', DIFF_FEATS)

clf_xgb = xgb.XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                              eval_metric='logloss', random_state=42, n_jobs=-1)
clf_xgb.fit(X_tr_c, y_tr_c, eval_set=[(X_te_c, y_te_c)], verbose=False)

proba_xgb = clf_xgb.predict_proba(X_te_c)[:, 1]
acc_xgb   = accuracy_score(y_te_c, (proba_xgb > 0.5).astype(int))
auc_xgb   = roc_auc_score(y_te_c, proba_xgb)
brier_xgb = brier_score_loss(y_te_c, proba_xgb)
print(f"XGBoost  |  Acc: {acc_xgb:.3f}  AUC: {auc_xgb:.3f}  Brier: {brier_xgb:.4f}")

## Section 4b — Win Probability (FastAI Tabular)

In [ ]:
try:
    from fastai.tabular.all import *

    feat_df = game_matrix[DIFF_FEATS + ['win','season']].dropna().copy().reset_index(drop=True)
    feat_df['win'] = feat_df['win'].astype(str)
    valid_idx = feat_df[feat_df['season'] == TEST_YEAR].index.tolist()
    if len(valid_idx) == 0:
        valid_idx = feat_df.index[-max(1, len(feat_df)//10):].tolist()

    to = TabularDataLoaders.from_df(
        feat_df, path='.', y_names='win', y_block=CategoryBlock(),
        cat_names=[], cont_names=DIFF_FEATS,
        procs=[FillMissing, Normalize], valid_idx=valid_idx, bs=256)

    learn = tabular_learner(to, layers=[200, 100], metrics=accuracy)
    learn.fit_one_cycle(10, 1e-3)

    preds_fa, targs_fa = learn.get_preds(dl=to.valid)
    proba_fastai = preds_fa[:, 1].numpy()
    y_fa_valid   = (np.array(targs_fa) == 1).astype(int)
    acc_fa   = accuracy_score(y_fa_valid, (proba_fastai > 0.5).astype(int))
    auc_fa   = roc_auc_score(y_fa_valid, proba_fastai)
    brier_fa = brier_score_loss(y_fa_valid, proba_fastai)
    _fastai_ok = True
    print(f"FastAI  |  Acc: {acc_fa:.3f}  AUC: {auc_fa:.3f}  Brier: {brier_fa:.4f}")
except Exception as e:
    print(f"FastAI skipped: {e}")
    proba_fastai = proba_xgb
    y_fa_valid   = y_te_c
    _fastai_ok   = False

## Section 4c — Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for proba, y_true, label, color in [
    (proba_xgb,   y_te_c,     'XGBoost', 'royalblue'),
    (proba_fastai, y_fa_valid, 'FastAI',  'darkorange'),
]:
    fp, mp = calibration_curve(y_true, proba, n_bins=10, strategy='uniform')
    axes[0].plot(mp, fp, marker='o', label=label, color=color)
axes[0].plot([0,1],[0,1],'k--', label='Perfect')
axes[0].set_title('Calibration Curves'); axes[0].legend()

metrics = pd.DataFrame({
    'Model': ['XGBoost','FastAI'],
    'Acc':   [acc_xgb,  acc_fa  if _fastai_ok else acc_xgb],
    'AUC':   [auc_xgb,  auc_fa  if _fastai_ok else auc_xgb],
}).set_index('Model')
metrics.plot(kind='bar', ax=axes[1], colormap='Set2')
axes[1].set_title('Model Metrics'); axes[1].set_ylim(0.4, 1.0)
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## Section 5 — Stacked Ensemble with Isotonic Calibration

In [ ]:
from sklearn.isotonic import IsotonicRegression

# Align FastAI predictions to XGBoost test length before stacking
_fa_for_stack = proba_fastai if len(proba_fastai) == len(proba_xgb) else proba_xgb
meta_X = np.column_stack([proba_xgb, _fa_for_stack, preds_reg])
meta_y = y_te_c

meta_lr = LogisticRegression(C=1.0, max_iter=500)
meta_lr.fit(meta_X, meta_y)
proba_ensemble = meta_lr.predict_proba(meta_X)[:, 1]

iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(proba_ensemble, meta_y)
proba_calibrated = iso.predict(proba_ensemble)

acc_ens   = accuracy_score(meta_y, (proba_calibrated > 0.5).astype(int))
auc_ens   = roc_auc_score(meta_y, proba_calibrated)
brier_ens = brier_score_loss(meta_y, proba_calibrated)
print(f"Ensemble  |  Acc: {acc_ens:.3f}  AUC: {auc_ens:.3f}  Brier: {brier_ens:.4f}")
print(f"Weights   |  XGB: {meta_lr.coef_[0][0]:.3f}  FA: {meta_lr.coef_[0][1]:.3f}  RD: {meta_lr.coef_[0][2]:.3f}")

## Section 6 — Game Predictor Tool

### How to use

**Step 1 — Find the exact team name**
Run the rankings cell above. It prints a numbered list like:
```
 #1  Tennessee Volunteers
 #2  LSU Tigers
```
Copy the name exactly as shown — spacing and capitalization matter.

**Step 2 — Call `predict_game()`**
```python
predict_game("Tennessee Volunteers", "LSU Tigers", is_home_a=True)
```
| Argument | Description |
|----------|-------------|
| `team_a` | First team (home by default) |
| `team_b` | Second team (away by default) |
| `is_home_a` | `True` if team_a is at home, `False` for neutral site or away |
| `year` | Season to pull stats from (default: 2026) |

**Step 3 — Read the output**
- **Ensemble win prob** — blended XGBoost + FastAI + run-diff prediction (most reliable)
- **XGBoost win prob** — standalone classifier
- **Elo win prob** — based purely on head-to-head history
- **Predicted run diff** — positive = team_a wins by that margin

**Step 4 — Predict multiple games at once**
```python
predict_games_table([
    ("Tennessee Volunteers", "LSU Tigers",       True),
    ("Florida Gators",       "Arkansas Razorbacks", False),
])
```
Returns a formatted table with win probabilities and predicted margins for all matchups.

In [ ]:
def predict_game(team_a, team_b, is_home_a=True, year=TEST_YEAR, verbose=True):
    s_idx  = team_stats[team_stats['season'] == year].set_index('team')
    elo_lk = elo_df.set_index('team')['elo']

    def get(t): return s_idx.loc[t] if t in s_idx.index else pd.Series(dtype=float)
    sa, sb = get(team_a), get(team_b)

    row = {'is_home': int(is_home_a), 'neutral': 0}
    for c in FEAT_COLS:
        row[f'd_{c}'] = sa.get(c, np.nan) - sb.get(c, np.nan)

    X = np.array([[row.get(f, 0) for f in DIFF_FEATS]])
    rd_pred = float(reg.predict(X)[0])
    wp_xgb  = float(clf_xgb.predict_proba(X)[0, 1])
    try:
        meta_in = np.array([[wp_xgb, wp_xgb, rd_pred]])
        wp_ens  = float(iso.predict(meta_lr.predict_proba(meta_in)[:, 1])[0])
    except Exception:
        wp_ens = wp_xgb

    ea = elo_lk.get(team_a, ELO_INIT) + (ELO_HOME if is_home_a else 0)
    eb = elo_lk.get(team_b, ELO_INIT)
    wp_elo = expected_win(ea, eb)

    if verbose:
        side = 'Home' if is_home_a else 'Away/Neutral'
        fav  = team_a if wp_ens > 0.5 else team_b
        print(f"\n{'='*52}")
        print(f"  {team_a} ({side}) vs {team_b}")
        print(f"{'='*52}")
        print(f"  Ensemble win prob : {wp_ens:.1%}  [favors {fav}]")
        print(f"  XGBoost win prob  : {wp_xgb:.1%}")
        print(f"  Elo win prob      : {wp_elo:.1%}")
        print(f"  Predicted run diff: {rd_pred:+.1f}")
        print(f"  Elo ratings       : {elo_lk.get(team_a,ELO_INIT):.0f} vs {elo_lk.get(team_b,ELO_INIT):.0f}")
    return {'team_a':team_a,'team_b':team_b,'win_prob':wp_ens,'run_diff':rd_pred,'elo_wp':wp_elo}

# Edit team names to match exactly what appears in the Top 25 rankings above
sample = [('Tennessee Volunteers','Arkansas Razorbacks',True),
          ('LSU Tigers','Florida Gators',False),
          ('Texas Longhorns','Oklahoma State Cowboys',True)]
for a, b, home in sample:
    try: predict_game(a, b, is_home_a=home)
    except Exception as e: print(f"{a} vs {b}: {e}")

In [ ]:
def predict_games_table(matchups, year=TEST_YEAR):
    rows = []
    for a, b, home in matchups:
        try:
            r = predict_game(a, b, is_home_a=home, year=year, verbose=False)
            rows.append({'Home/Neutral': a, 'Away': b,
                         'Win Prob (A)': f"{r['win_prob']:.1%}",
                         'Pred Margin': f"{r['run_diff']:+.1f}",
                         'Elo WP': f"{r['elo_wp']:.1%}"})
        except Exception as e:
            rows.append({'Home/Neutral': a, 'Away': b, 'Error': str(e)})
    return pd.DataFrame(rows)

# NCAA Tournament bracket — edit with actual matchups from the rankings above
upcoming = [('Tennessee Volunteers','Arkansas Razorbacks',True),
            ('LSU Tigers','Florida Gators',False)]
predict_games_table(upcoming)

## Section 7 — Betting Lines (Odds API)

In [ ]:
if IN_COLAB:
    try:
        from google.colab import userdata
        ODDS_API_KEY = (userdata.get('ODDS_API_KEY') or '').strip()
    except Exception:
        ODDS_API_KEY = ''
else:
    ODDS_API_KEY = (os.environ.get('ODDS_API_KEY') or '').strip()

ODDS_API_BASE = 'https://api.the-odds-api.com/v4'
SPORT_KEY     = None   # discovered below

def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return 100/(100+o) if o > 0 else abs(o)/(abs(o)+100)

if not ODDS_API_KEY:
    hint = 'Colab Secrets sidebar (key icon) -> ODDS_API_KEY' if IN_COLAB else '.env file'
    print(f"ODDS_API_KEY not set -- add it to {hint}")
else:
    print(f"Key loaded ({len(ODDS_API_KEY)} chars). Checking available sports ...")
    _r = requests.get(ODDS_API_BASE + '/sports',
                      params=dict(apiKey=ODDS_API_KEY), timeout=10)
    if _r.status_code == 401:
        print("401 Unauthorized -- key is wrong or expired.")
        print("Go to https://the-odds-api.com/account/ and copy your key fresh.")
    elif _r.status_code != 200:
        print(f"Sports list error {_r.status_code}: {_r.text[:300]}")
    else:
        _sports  = _r.json()
        _baseball = [s for s in _sports
                     if 'baseball' in s.get('key','').lower()
                     or 'baseball' in s.get('title','').lower()]
        print("Baseball sports on your plan:")
        for s in _baseball:
            print(f"  {s['key']:40s} {s['title']}")
        for _cand in ['baseball_ncaa','baseball_college','baseball_ncaab']:
            if any(s['key'] == _cand for s in _baseball):
                SPORT_KEY = _cand; break
        if not SPORT_KEY and _baseball:
            SPORT_KEY = _baseball[0]['key']
        print(f"\nUsing sport key : {SPORT_KEY}")
        print(f"Requests remaining: {_r.headers.get('x-requests-remaining','unknown')}")

In [ ]:
def fetch_live_odds():
    if not ODDS_API_KEY or not SPORT_KEY:
        print("Key or sport key missing -- run the cell above first."); return pd.DataFrame()
    _r = requests.get(f"{ODDS_API_BASE}/sports/{SPORT_KEY}/odds",
                      params=dict(apiKey=ODDS_API_KEY, regions='us',
                                  markets='h2h,spreads', oddsFormat='american'),
                      timeout=10)
    if _r.status_code != 200:
        print(f"Odds fetch error {_r.status_code}: {_r.text[:300]}"); return pd.DataFrame()
    rows = []
    for game in _r.json():
        home, away = game['home_team'], game['away_team']
        raw_time = game.get('commence_time', '')
        try:
            from datetime import datetime, timezone
            dt = datetime.fromisoformat(raw_time.replace('Z', '+00:00'))
            dt_et = dt.astimezone(timezone.utc)  # keep UTC; display as ET below
            game_date = dt.strftime('%b %-d')
            game_time = dt.strftime('%I:%M %p UTC')
        except Exception:
            game_date = raw_time[:10]
            game_time = ''
        for bkm in game.get('bookmakers', [])[:1]:
            for mkt in bkm.get('markets', []):
                if mkt['key'] == 'h2h':
                    out = {o['name']: o['price'] for o in mkt['outcomes']}
                    rows.append({'date': game_date, 'time': game_time,
                                 'home': home, 'away': away,
                                 'ml_home': out.get(home), 'ml_away': out.get(away)})
    print(f"Requests remaining: {_r.headers.get('x-requests-remaining','unknown')}")
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df[['date', 'time', 'home', 'away', 'ml_home', 'ml_away']]
    return df

live_odds = fetch_live_odds()
live_odds

## Section 7b — Best Bets Tool

Query by **date**, **teams**, or **budget** to get ranked betting recommendations with expected value and Kelly sizing.

```python
# All today's games with positive edge
best_bets()

# Filter by date string
best_bets(date='May 15')

# Filter to specific teams
best_bets(teams=['Tennessee Volunteers', 'LSU Tigers'])

# With a $500 budget — shows exact $ to bet per game
best_bets(budget=500)

# Combine filters
best_bets(date='May 15', budget=200, edge_min=0.05)
```

**Columns:**
| Column | Meaning |
|--------|---------|
| `bet_on` | Which team to bet |
| `ml` | Current American moneyline |
| `model_wp` | Model's estimated win probability |
| `implied_prob` | Vegas implied probability from the line |
| `edge` | model_wp − implied_prob (your statistical advantage) |
| `ev_per_unit` | Expected value per $1 wagered (positive = profitable long-run) |
| `kelly_pct` | Recommended bankroll % (quarter-Kelly, 10% cap) |
| `bet_$` | Dollar amount to bet (only shown when budget is provided) |


In [ ]:
def best_bets(date=None, teams=None, budget=None, edge_min=0.03, year=TEST_YEAR):
    """
    Returns ranked betting recommendations from live odds.

    Args:
        date    : filter by date string (e.g., 'May 15') — partial match, case-insensitive
        teams   : list of team names to include, or None for all games
        budget  : total $ budget to allocate; triggers bet_$ column showing Kelly-sized amounts
        edge_min: minimum model edge required (default 0.03 = 3%)
        year    : season for model stats (default TEST_YEAR)

    Returns:
        DataFrame sorted by expected value (best bets first)
    """
    from IPython.display import display as ipy_display

    odds = fetch_live_odds()
    if odds.empty:
        print("No live odds available — check ODDS_API_KEY and run Section 7 first.")
        return pd.DataFrame()

    if date:
        odds = odds[odds['date'].str.contains(str(date), case=False, na=False)]
    if teams:
        mask = odds['home'].isin(teams) | odds['away'].isin(teams)
        odds = odds[mask]
    if odds.empty:
        print("No games match your filters."); return pd.DataFrame()

    rows = []
    for _, g in odds.iterrows():
        home, away = g['home'], g['away']
        ml_home = g.get('ml_home')
        ml_away = g.get('ml_away')
        try:
            pred = predict_game(home, away, is_home_a=True, year=year, verbose=False)
        except Exception:
            continue
        wp_home = pred['win_prob']
        wp_away = 1.0 - wp_home

        for side, team, ml, model_wp in [
            ('Home', home, ml_home, wp_home),
            ('Away', away, ml_away, wp_away),
        ]:
            if pd.isna(ml) or ml is None: continue
            ml = float(ml)
            implied = american_to_prob(ml)
            edge = model_wp - implied
            if edge < edge_min: continue

            b = (100 / abs(ml)) if ml < 0 else (ml / 100)
            ev = model_wp * b - (1.0 - model_wp)   # EV per $1 wagered
            kf = kelly_fraction(model_wp, odds=ml)

            rows.append({
                'date':         g.get('date', ''),
                'time':         g.get('time', ''),
                'matchup':      f"{home} vs {away}",
                'bet_on':       team,
                'side':         side,
                'ml':           int(ml),
                'model_wp':     model_wp,
                'implied_prob': implied,
                'edge':         edge,
                'ev_per_unit':  ev,
                'kelly_pct':    kf,
                'bet_$':        round(budget * kf, 2) if budget else None,
            })

    if not rows:
        print(f"No bets found with edge > {edge_min:.0%}. Try lowering edge_min.")
        return pd.DataFrame()

    df = pd.DataFrame(rows).sort_values('ev_per_unit', ascending=False).reset_index(drop=True)

    show = ['date', 'time', 'matchup', 'bet_on', 'side', 'ml',
            'model_wp', 'implied_prob', 'edge', 'ev_per_unit', 'kelly_pct']
    if budget:
        show.append('bet_$')

    fmt = {
        'model_wp':     '{:.1%}',
        'implied_prob': '{:.1%}',
        'edge':         '{:.1%}',
        'ev_per_unit':  '{:+.3f}',
        'kelly_pct':    '{:.1%}',
    }
    if budget:
        fmt['bet_$'] = '${:.2f}'

    budget_str = f"  |  budget: ${budget:,.0f}" if budget else ""
    print(f"\n{'='*65}")
    print(f"  BEST BETS  |  {len(df)} edge{'s' if len(df)!=1 else ''} found"
          f"  |  min edge: {edge_min:.0%}{budget_str}")
    print(f"{'='*65}")
    ipy_display(df[show].style.format(fmt)
                .bar(subset=['ev_per_unit'], color=['#d65f5f','#5fba7d'], align='zero')
                .bar(subset=['edge'], color='#5fba7d'))

    if budget:
        total = df['bet_$'].sum()
        print(f"\n  Total allocated: ${total:,.2f} / ${budget:,.0f}  |  "
              f"Remaining: ${budget - total:,.2f}")

    return df

# ── example calls (edit to match your query) ──────────────────────────────────
# best_bets()                                    # all today's games
# best_bets(date='May 15', budget=500)           # filter by date + budget
# best_bets(teams=['Tennessee Volunteers'], budget=200, edge_min=0.04)
print("best_bets() loaded. Call it after running Section 7 to fetch live odds.")


## Section 8 — Backtesting

In [ ]:
def backtest(matrix, model_proba, year=None, edge_threshold=0.03):
    df = matrix.copy()
    if year: df = df[df['season'] == year]
    df = df.dropna(subset=['win'] + DIFF_FEATS).copy()
    if model_proba is not None and len(model_proba) == len(df):
        df['model_wp'] = model_proba
    if 'model_wp' not in df.columns:
        print("model_wp not set — run ensemble section first"); return
    df['implied_prob'] = 0.524  # -110 juice
    df['edge'] = df['model_wp'] - df['implied_prob']
    bets = df[df['edge'] > edge_threshold].copy()
    if len(bets) == 0:
        print(f"No bets at edge > {edge_threshold:.0%}"); return
    bets['payout'] = bets['win'].apply(lambda w: 100/110 if w == 1 else -1)
    pnl    = bets['payout'].sum()
    roi    = pnl / len(bets)
    record = f"{int(bets['win'].sum())}-{int((1-bets['win']).sum())}"
    print(f"Bets: {len(bets)}  Record: {record}  P&L: {pnl:+.2f}u  ROI: {roi:+.1%}")
    return bets

test_rows = game_matrix[game_matrix['season'] == TEST_YEAR].dropna(subset=['win']+DIFF_FEATS).copy()
if len(test_rows) == len(proba_calibrated):
    test_rows['model_wp'] = proba_calibrated
    backtest(test_rows, None, edge_threshold=0.03)
else:
    print(f"Length mismatch: {len(test_rows)} rows vs {len(proba_calibrated)} proba")

## Section 9 — Kelly Criterion Bankroll Simulation

In [ ]:
def kelly_fraction(wp, odds=-110, frac=KELLY_FRACTION, cap=KELLY_CAP):
    imp = american_to_prob(odds)
    if wp <= imp: return 0.0
    b = (100/abs(odds)) if odds < 0 else (odds/100)
    return min(max(frac * (wp*(b+1)-1)/b, 0), cap)

def simulate_bankroll(bets_df, br=STARTING_BANKROLL, frac=KELLY_FRACTION, cap=KELLY_CAP, odds=-110):
    history = [br]
    for _, row in bets_df.iterrows():
        f = kelly_fraction(row.get('model_wp', 0.5), odds=odds, frac=frac, cap=cap)
        b = (100/abs(odds)) if odds < 0 else (odds/100)
        br += br*f*b if row.get('win',0) == 1 else -br*f
        history.append(br)
    return br, (br - STARTING_BANKROLL)/STARTING_BANKROLL, history

if 'model_wp' in test_rows.columns:
    bets = test_rows[test_rows['model_wp'] > 0.55].copy()
    final, roi, hist = simulate_bankroll(bets)
    print(f"Bets: {len(bets)}  Final: ${final:,.2f}  ROI: {roi:+.1%}")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(hist, color='royalblue', linewidth=1.5)
    ax.axhline(STARTING_BANKROLL, color='gray', linestyle='--', linewidth=0.8)
    ax.fill_between(range(len(hist)), STARTING_BANKROLL, hist, alpha=0.15, color='royalblue')
    ax.set_title(f'{TEST_YEAR} Kelly Bankroll Simulation (Quarter Kelly)')
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
    plt.tight_layout(); plt.show()

In [ ]:
# Scenario explorer: sweep edge threshold x Kelly fraction
def scenario_explorer(df, thresholds=None, fractions=None):
    if thresholds is None: thresholds = [0.02, 0.04, 0.06, 0.08, 0.10]
    if fractions  is None: fractions  = [0.10, 0.25, 0.50]
    rows = []
    df['edge'] = df.get('model_wp', 0.5) - 0.524
    for t in thresholds:
        sub = df[df['edge'] > t]
        for f in fractions:
            if len(sub) == 0:
                rows.append({'Edge':f'{t:.0%}','Kelly':f'{f:.0%}','Bets':0,'ROI':'N/A','Final':'N/A'})
                continue
            _, roi, hist = simulate_bankroll(sub, frac=f)
            rows.append({'Edge':f'{t:.0%}','Kelly':f'{f:.0%}',
                         'Bets':len(sub),'ROI':f'{roi:+.1%}','Final':f'${hist[-1]:,.0f}'})
    return pd.DataFrame(rows)

if 'model_wp' in test_rows.columns:
    scenario_explorer(test_rows)

## Section 10 — Interactive Dashboard (Inline)

The full dashboard — rankings, team profiles, comparison radar, and game predictor — is embedded below.
Also available as a standalone link: [trevmon28.github.io/CollegeBaseballAnalytics](https://trevmon28.github.io/CollegeBaseballAnalytics/)

In [ ]:
from IPython.display import IFrame, HTML, display

DASHBOARD_URL = 'https://trevmon28.github.io/CollegeBaseballAnalytics/'

try:
    import google.colab
    display(HTML(
        '<div style="border:1px solid #1a2f52;border-radius:10px;overflow:hidden">'
        '<iframe src="' + DASHBOARD_URL + '" width="100%" height="820" '
        'style="border:none;background:#07090f" allowfullscreen></iframe></div>'
    ))
except ImportError:
    display(IFrame(DASHBOARD_URL, width='100%', height=820))
    print('Dashboard also at http://127.0.0.1:8050 (if dashboard.py is running)')


## Section 10b — Database Query Interface

Query the local game-results database by team, date, season, or result.

```python
# All Tennessee Volunteers games in 2025
query_db(team='Tennessee', season=2025)

# Head-to-head: Tennessee vs Arkansas
query_db(team='Tennessee', opponent='Arkansas')

# Tennessee home wins only
query_db(team='Tennessee', home_only=True, result='W')

# Date range
query_db(date_from='2026-04-01', date_to='2026-05-15')

# Season summary for a team
team_stats_query('LSU Tigers', season=2025)
```


In [ ]:
def query_db(team=None, opponent=None, season=None, date_from=None, date_to=None,
             home_only=False, result=None, limit=50):
    """
    Query game results.

    Args:
        team      : partial team name match (case-insensitive), e.g. 'Tennessee'
        opponent  : filter to specific opponent (partial match)
        season    : int or list of ints, e.g. 2025 or [2024, 2025]
        date_from : 'YYYY-MM-DD' string (inclusive)
        date_to   : 'YYYY-MM-DD' string (inclusive)
        home_only : only games where `team` was the home team
        result    : 'W' or 'L' (from the perspective of `team`)
        limit     : max rows returned (default 50)

    Returns:
        DataFrame of matching games, sorted newest first
    """
    df = games.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    if season is not None:
        seasons = [season] if isinstance(season, int) else list(season)
        df = df[df['season'].isin(seasons)]

    if date_from:
        df = df[df['date'] >= pd.to_datetime(date_from)]
    if date_to:
        df = df[df['date'] <= pd.to_datetime(date_to)]

    if team:
        if home_only:
            df = df[df['home_team'].str.contains(team, case=False, na=False)]
        else:
            mask = (df['home_team'].str.contains(team, case=False, na=False) |
                    df['away_team'].str.contains(team, case=False, na=False))
            df = df[mask]

        if opponent:
            opp_mask = (df['home_team'].str.contains(opponent, case=False, na=False) |
                        df['away_team'].str.contains(opponent, case=False, na=False))
            df = df[opp_mask]

        if result:
            team_won = (
                (df['home_team'].str.contains(team, case=False, na=False) & (df['home_score'] > df['away_score'])) |
                (df['away_team'].str.contains(team, case=False, na=False) & (df['away_score'] > df['home_score']))
            )
            df = df[team_won] if result.upper() == 'W' else df[~team_won]
    elif opponent:
        mask = (df['home_team'].str.contains(opponent, case=False, na=False) |
                df['away_team'].str.contains(opponent, case=False, na=False))
        df = df[mask]

    df = df.sort_values('date', ascending=False).head(limit).reset_index(drop=True)
    df['run_diff'] = (df['home_score'] - df['away_score']).apply(lambda x: f"{x:+.0f}")
    df['result_str'] = df.apply(
        lambda r: f"{r['home_team']} {r['home_score']:.0f}-{r['away_score']:.0f} {r['away_team']}", axis=1
    )
    print(f"Found {len(df)} games (showing up to {limit}).")
    return df[['season','date','home_team','home_score','away_team','away_score','run_diff','neutral']]


def team_stats_query(team, season=None):
    """Season-by-season summary stats for a team."""
    df = team_stats.copy()
    mask = df['team'].str.contains(team, case=False, na=False)
    df = df[mask]
    if season:
        df = df[df['season'] == season]
    show = ['season','team','games','wins','losses','win_pct',
            'avg_runs_scored','avg_runs_allowed','avg_run_diff','pythagorean_win_pct']
    show = [c for c in show if c in df.columns]
    fmt = {
        'win_pct':'{:.3f}','avg_runs_scored':'{:.2f}',
        'avg_runs_allowed':'{:.2f}','avg_run_diff':'{:+.2f}',
        'pythagorean_win_pct':'{:.3f}',
    }
    print(f"Season stats for '{team}':")
    from IPython.display import display as ipy_display
    ipy_display(df[show].sort_values('season').style.format(fmt))
    return df[show]


# ── sample queries ─────────────────────────────────────────────────────────────
print("query_db() and team_stats_query() loaded.")
print("Examples:")
print("  query_db(team='Tennessee', season=2025)")
print("  query_db(team='LSU', opponent='Arkansas', result='W')")
print("  team_stats_query('Tennessee Volunteers')")
